## Klasifikacija na skupu podataka `diabetes.csv`

Ispravljena verzija bez curenja podataka (*data leakage*). U originalnom rešenju koje je bilo na sajtu kursa, proseci za imputaciju i granice za outlier-e računati su nad celim skupom pre podele na trening i test skup. Ovde se sve statistike računaju isključivo na trening skupu, a zatim se primenjuju na test skup.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


**a)** Prikazati osnovne deskriptivne statistike (prosek, standardna devijacija, kvartili).

In [3]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


**b)** Ukoliko ima nedostajućih vrednosti, gde za kolone `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` vrednost 0 označava nedostajuću vrednost, zameniti ih prosekom za tu kolonu.

Nule prvo samo obeležavamo kao `NaN`, jer to ne koristi nikakvu statistiku iz podataka. Podela na trening i test skup mora da se uradi pre računanja proseka za imputaciju.

In [4]:
missing_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

(df[missing_columns] == 0).sum()

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

In [5]:
X = df.drop(columns='Outcome')
y = df['Outcome']

In [6]:
import numpy as np

In [7]:
X[missing_columns] = X[missing_columns].replace(0, np.nan)

In [8]:
X.isna().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

Podatke delimo pre imputacije, uklanjanja outlier-a i skaliranja. Parametar `stratify=y` čuva odnos klasa u trening i test skupu.

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.3,
    random_state=123,
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((537, 8), (231, 8), (537,), (231,))

In [11]:
train_means = X_train[missing_columns].mean()
train_means

Glucose          121.374532
BloodPressure     72.517578
SkinThickness     29.008043
Insulin          152.429603
BMI               32.573055
dtype: float64

In [12]:
X_train[missing_columns] = X_train[missing_columns].fillna(train_means)
X_test[missing_columns] = X_test[missing_columns].fillna(train_means)

**c)** Odrediti elemente van granica (eng. outliers) korišćenjem standardne devijacije. Ukoliko postoje, zameniti ih graničnim vrednostima.

In [13]:
X_train.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000
mean,3.873371,121.374532,72.517578,29.008043,152.429603,32.573055,0.475408,33.296089
std,3.441784,30.809934,12.104785,8.656481,83.264749,7.110091,0.331440,12.018015
min,0.000000,44.000000,30.000000,7.000000,14.000000,18.200000,0.078000,21.000000
25%,1.000000,99.000000,64.000000,25.000000,120.000000,27.500000,0.240000,24.000000
50%,3.000000,117.000000,72.000000,29.008043,152.429603,32.400000,0.378000,29.000000
75%,6.000000,141.000000,80.000000,32.000000,152.429603,36.600000,0.637000,41.000000
max,17.000000,199.000000,114.000000,63.000000,846.000000,67.100000,2.420000,81.000000


In [14]:
train_feature_means = X_train.mean()
train_feature_stds = X_train.std()

lower_bounds = train_feature_means - 3 * train_feature_stds
upper_bounds = train_feature_means + 3 * train_feature_stds

X_train = X_train.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
X_test = X_test.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

In [15]:
X_train.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
count,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000,537.000000
mean,3.866662,121.374532,72.528882,28.981846,149.238658,32.529877,0.469581,33.268250
std,3.419239,30.809934,11.991251,8.567480,67.695076,6.949894,0.307414,11.922820
min,0.000000,44.000000,36.203223,7.000000,14.000000,18.200000,0.078000,21.000000
25%,1.000000,99.000000,64.000000,25.000000,120.000000,27.500000,0.240000,24.000000
50%,3.000000,117.000000,72.000000,29.008043,152.429603,32.400000,0.378000,29.000000
75%,6.000000,141.000000,80.000000,32.000000,152.429603,36.600000,0.637000,41.000000
max,14.198722,199.000000,108.831934,54.977485,402.223850,53.903327,1.469728,69.350135


**d)** Klasifikovati podatke (ciljna kolona je `Outcome`) korišćenjem SVM algoritma. Da li je potrebno podeliti podatke na trening i test skup? Da li je potrebno primeniti neki vid normalizacije podataka? Zašto?

Kao i uvek kod nadgledanog učenja, potrebno je podeliti podatke na trening i test skup da bismo model ocenili na podacima koje nije video tokom obučavanja. Potrebna je i normalizacija, jer je SVM osetljiv na skalu atributa.

In [16]:
from sklearn.preprocessing import StandardScaler

In [17]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
from sklearn.svm import SVC

In [19]:
model_svm = SVC()
model_svm.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


**e)** Prikazati matricu konfuzije, tačnost i F1. Oceniti kvalitet modela.

In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

In [21]:
y_train_pred = model_svm.predict(X_train)

In [22]:
confusion_matrix(y_train, y_train_pred)

array([[324,  26],
       [ 69, 118]])

In [23]:
accuracy_score(y_train, y_train_pred)

0.8230912476722533

In [24]:
f1_score(y_train, y_train_pred)

0.7129909365558912

In [25]:
y_test_pred = model_svm.predict(X_test)

In [26]:
confusion_matrix(y_test, y_test_pred)

array([[134,  16],
       [ 40,  41]])

In [27]:
accuracy_score(y_test, y_test_pred)

0.7575757575757576

In [28]:
f1_score(y_test, y_test_pred)

0.5942028985507246

**f)** Uporediti rezultate linearnog i kernelizovanog SVM-a.

In [29]:
from sklearn.svm import LinearSVC

In [30]:
model_lin_svm = LinearSVC()
model_lin_svm.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [31]:
y_train_pred = model_lin_svm.predict(X_train)

In [32]:
confusion_matrix(y_train, y_train_pred)

array([[315,  35],
       [ 78, 109]])

In [33]:
accuracy_score(y_train, y_train_pred)

0.7895716945996276

In [34]:
f1_score(y_train, y_train_pred)

0.6586102719033232

In [35]:
y_test_pred = model_lin_svm.predict(X_test)

In [36]:
confusion_matrix(y_test, y_test_pred)

array([[131,  19],
       [ 41,  40]])

In [37]:
accuracy_score(y_test, y_test_pred)

0.7402597402597403

In [38]:
f1_score(y_test, y_test_pred)

0.5714285714285714

**g)** Da li je skup balansiran? Ukoliko nije, primeniti *SMOTE* i obučiti novi model na izbalansiranom skupu. Uporediti rezultate sa originalnim modelom.

In [39]:
y.value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [40]:
y_train.value_counts()

Outcome
0    350
1    187
Name: count, dtype: int64

Skup podataka je blago nebalansiran, pa možemo da sprovedemo eksperiment sa balansiranjem uz pomoć SMOTE.

In [41]:
from imblearn.over_sampling import SMOTE

In [42]:
smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [43]:
y_train_resampled.value_counts()

Outcome
1    350
0    350
Name: count, dtype: int64

In [44]:
model_svm_smote = SVC()
model_svm_smote.fit(X_train_resampled, y_train_resampled)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [45]:
y_test_pred = model_svm_smote.predict(X_test)

In [46]:
confusion_matrix(y_test, y_test_pred)

array([[118,  32],
       [ 21,  60]])

In [47]:
accuracy_score(y_test, y_test_pred)

0.7705627705627706

In [48]:
f1_score(y_test, y_test_pred)

0.6936416184971098

Tačnost na test skupu je slična kao i ranije, ali f1 mera je sada značajno veća, odnosno SMOTE je poboljšao performanse modela za manjinsku klasu, po cenu više grešaka kod većinske klase. 